In [11]:
import pandas as pd

from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import os
from sklearn.metrics import accuracy_score

In [14]:
# Unscaled datasets
feature_dir = '..\\feature_engineering'
train_unscaled_path = os.path.join(feature_dir, 'train_unscaled_processed.csv')
test_unscaled_path = os.path.join(feature_dir, 'test_unscaled_processed.csv')
train_unscaled = pd.read_csv(train_unscaled_path)
test_unscaled = pd.read_csv(test_unscaled_path)

# Scaled datasets
train_scaled_path = os.path.join(feature_dir, 'train_scaled_processed.csv')
test_scaled_path = os.path.join(feature_dir, 'test_scaled_processed.csv')
train_scaled = pd.read_csv(train_scaled_path)
test_scaled = pd.read_csv(test_scaled_path)

In [15]:
target_cols = ["Fatigue", "FutureHealthRisk", "DiabetesRisk", "AnemiaRisk", "PCOSRisk"]

In [16]:
# Unscaled
X_train = train_unscaled.drop(columns=target_cols)
y_train = train_unscaled[target_cols]

X_test = test_unscaled.drop(columns=target_cols)
y_test = test_unscaled[target_cols]

# Scaled (for LR)
X_train_scaled = train_scaled.drop(columns=target_cols)
X_test_scaled = test_scaled.drop(columns=target_cols)

In [17]:
rf_model = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, random_state=42)
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [18]:
lr_model = MultiOutputClassifier(
    LogisticRegression(max_iter=2000)
)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

In [19]:
xgb_model = MultiOutputClassifier(
    XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric='mlogloss'
    )
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

In [20]:
def evaluate_model(name, y_test, y_pred):
    print(f"\n{name} Evaluation:")

    for i, col in enumerate(target_cols):
        acc = accuracy_score(y_test[col], y_pred[:, i])
        print(f"{col} Accuracy: {acc:.4f}")

# Evaluate all
evaluate_model("Random Forest", y_test, rf_pred)
evaluate_model("Logistic Regression", y_test, lr_pred)
evaluate_model("XGBoost", y_test, xgb_pred)


Random Forest Evaluation:
Fatigue Accuracy: 0.9221
FutureHealthRisk Accuracy: 0.9464
DiabetesRisk Accuracy: 0.9357
AnemiaRisk Accuracy: 0.9736
PCOSRisk Accuracy: 0.9500

Logistic Regression Evaluation:
Fatigue Accuracy: 0.7407
FutureHealthRisk Accuracy: 0.9914
DiabetesRisk Accuracy: 0.9900
AnemiaRisk Accuracy: 0.9929
PCOSRisk Accuracy: 0.9893

XGBoost Evaluation:
Fatigue Accuracy: 0.9636
FutureHealthRisk Accuracy: 0.9629
DiabetesRisk Accuracy: 0.9536
AnemiaRisk Accuracy: 0.9821
PCOSRisk Accuracy: 0.9629


✅ Feature Engineering + Scaler saved successfully!


In [21]:
import joblib

joblib.dump(xgb_model, "multi_output_xgb_model.pkl")
print("✅ Multi-output model saved!")

✅ Multi-output model saved!


In [41]:
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(lr_model, "lr_model.pkl")
joblib.dump(xgb_model, "xgb_model.pkl")

['xgb_model.pkl']